In [1]:
%pip install ultralytics
import ultralytics
ultralytics.checks()

Ultralytics 8.4.27 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 43.6/112.6 GB disk)


In [22]:
import random
import cv2
from ultralytics import YOLO
from IPython.display import clear_output # Added import for clear_output
from google.colab.patches import cv2_imshow # Import cv2_imshow for Colab display

with open ('/content/coco.txt','r') as my_file:
  class_list = my_file.read().split('\n')
detection_color=[]
for _ in range(len(class_list)):
  r = random.randint(0,255)
  g = random.randint(0,255)
  b = random.randint(0,255)
  detection_color.append((r,g,b))

model = YOLO('/content/weights/yolov8n.pt')
cap=cv2.VideoCapture('/content/188613-883402208 (1).mp4')

if not cap.isOpened():
  print('cannot open video file')
  exit()

while True:
  ret,frame=cap.read()
  if not ret:
    print('video ended or failed')
    break

  detect_params = model.predict(source=[frame], conf=0.45, save=False)
  boxes = detect_params[0].boxes
  for i in range(len(boxes)):
    box=boxes[i]
    clsID = int(box.cls.cpu().numpy()[0])
    conf = box.conf.cpu().numpy()[0]
    bb = box.xyxy.cpu().numpy()[0]
    cv2.rectangle(
        frame, (int(bb[0]), int(bb[1])), (int(bb[2]), int(bb[3])),
        detection_color[clsID], 2)

    label=f'{class_list[clsID]}{round(conf*100,2)}%' # Moved label creation here
    cv2.putText(frame,label,(int(bb[0]),int(bb[1]-10)),cv2.FONT_HERSHEY_SIMPLEX,0.6,(255,255,255),2) # Moved putText here

  # Optional: Display the frame (requires setting up a display environment if running locally)
  #cv2.imshow('Object Detection', frame)
  #if cv2.waitKey(1) & 0xFF == ord('q'):
     #break
  clear_output(wait=True)
  cv2_imshow(frame) # Moved clear_output here
  cv2.waitKey(1) # This waits for a key press, might not be desired in automated execution

# Release the video capture object and close display windows
cap.release() # Moved to outside the loop
cv2.destroyAllWindows() # Moved to outside the loop

KeyboardInterrupt: 